In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from collections import Counter
import pandas as pd
import gensim
from scipy.stats import pearsonr, spearmanr


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
MAX_LEN = 20
EMBED_DIM = 300
HIDDEN_DIM = 50
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 0.001


In [4]:
class Vocabulary:
	def __init__(self, min_freq=1):
		self.word2idx = {"<PAD>": 0, "<UNK>": 1}
		self.idx2word = ["<PAD>", "<UNK>"]
		self.min_freq = min_freq

	def build_vocab(self, sentences: list[str]):
		counter = Counter()
		for sent in sentences:
			words = sent.split()
			counter.update(words)
		# Conservar palabras con min_freq
		for word, freq in counter.items():
			if freq >= self.min_freq:
				self.word2idx[word] = len(self.idx2word)
				self.idx2word.append(word)

	def __len__(self):
		return len(self.idx2word)

	def text_to_indices(self, text: str):
		tokens = text.split()
		return [self.word2idx.get(tok, self.word2idx["<UNK>"]) for tok in tokens]
	

In [5]:
def collate_fn(batch):
	sent1_pad = []
	sent2_pad = []
	len1_list = []
	len2_list = []
	labels = []

	max_len1 = max(item['len1'] for item in batch)
	max_len2 = max(item['len2'] for item in batch)

	for item in batch:
		padded1 = item['sent1'] + [0] * (max_len1 - item['len1'])
		sent1_pad.append(padded1)
		len1_list.append(item['len1'])
		
		padded2 = item['sent2'] + [0] * (max_len2 - item['len2'])
		sent2_pad.append(padded2)
		len2_list.append(item['len2'])

		labels.append(item['label'])

	return {
		'sent1': torch.tensor(sent1_pad, dtype=torch.long),
		'sent2': torch.tensor(sent2_pad, dtype=torch.long),
		'len1': torch.tensor(len1_list, dtype=torch.long),
		'len2': torch.tensor(len2_list, dtype=torch.long),
		'label': torch.stack(labels).unsqueeze(1)
	}

In [6]:
class SentencePairDataset(Dataset):
	def __init__(self, csv_file: str, vocab: Vocabulary, max_len: int):
		self.df = pd.read_csv(csv_file)
		self.vocab = vocab
		self.max_len = max_len

	def __len__(self):
		return len(self.df)

	def __getitem__(self, idx: int):
		row = self.df.iloc[idx]
		sent1 = row['sentence1']
		sent2 = row['sentence2']
		score = row['score_norm']

		indices1 = self.vocab.text_to_indices(sent1)
		indices2 = self.vocab.text_to_indices(sent2)

		# Truncar si es más largo que max_len
		if len(indices1) > self.max_len:
			indices1 = indices1[:self.max_len]
		if len(indices2) > self.max_len:
			indices2 = indices2[:self.max_len]

		return {
			'sent1': indices1,
			'sent2': indices2,
			'len1': len(indices1),
			'len2': len(indices2),
			'label': torch.tensor(score, dtype=torch.float32)
		}

In [7]:
def load_gensim_word2vec(path, vocab: Vocabulary, embed_dim: int):
	wv = gensim.models.KeyedVectors.load(path, mmap='r')
	embedding_matrix = np.random.uniform(-0.05, 0.05, (len(vocab), embed_dim))
	embedding_matrix[0] = 0
	found = 0
	for word, idx in vocab.word2idx.items():
		if word in wv.key_to_index:
			embedding_matrix[idx] = wv[word]
			found += 1
	print(f"Found {found}/{len(vocab)} words in word2vec model.")
	return embedding_matrix

In [8]:
class MaLSTM(nn.Module):
	def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pretrained_emb=None, freeze_emb=True):
		super().__init__()
		self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
		if pretrained_emb is not None:
			self.embedding.weight = nn.Parameter(torch.tensor(pretrained_emb, dtype=torch.float32))
			if freeze_emb:
				self.embedding.weight.requires_grad = False
		self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
		
	def forward_one(self, x, lengths):
		embedded  = self.embedding(x)
		packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
		lstm_out, (hidden, cell) = self.lstm(embedded)
		last_hidden = hidden.squeeze(0)
		return last_hidden

	def similarity(self, x, y):
		dist = torch.sum(torch.abs(x - y), dim=1, keepdim=True)
		return torch.exp(-dist)
	
	def forward(self, sent1, len1, sent2, len2):
		rep1 = self.forward_one(sent1, len1)
		rep2 = self.forward_one(sent2, len2)
		return self.similarity(rep1, rep2)

In [9]:
def train_model(model: nn.Module, train_loader, dev_loader, epochs, patience, lr):
	criterion = nn.MSELoss()
	optimizer = optim.Adam(model.parameters(), lr=lr)
	best_dev_loss = float('inf')
	patience_counter = 0

	for epoch in range(1, epochs+1):
		model.train()
		train_loss = 0.0
		for batch in train_loader:
			sent1 = batch['sent1'].to(device)
			len1 = batch['len1'].to(device)
			sent2 = batch['sent2'].to(device)
			len2 = batch['len2'].to(device)
			labels = batch['label'].to(device)

			optimizer.zero_grad()
			pred = model(sent1, len1, sent2, len2)
			loss = criterion(pred, labels)
			loss.backward()
			optimizer.step()

			train_loss += loss.item() * sent1.size(0)

		train_loss /= len(train_loader.dataset)

		# Validación
		model.eval()
		dev_loss = 0.0
		with torch.no_grad():
			for batch in dev_loader:
				sent1 = batch['sent1'].to(device)
				len1 = batch['len1'].to(device)
				sent2 = batch['sent2'].to(device)
				len2 = batch['len2'].to(device)
				labels = batch['label'].to(device)

				pred = model(sent1, len1, sent2, len2)
				loss = criterion(pred, labels)
				# Multiplicar por el batch size
				dev_loss += loss.item() * sent1.size(0)

		dev_loss /= len(dev_loader.dataset)

		print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.5f} | Dev Loss: {dev_loss:.5f}")

		# Early stopping
		if dev_loss < best_dev_loss:
			best_dev_loss = dev_loss
			patience_counter = 0
			torch.save(model.state_dict(), 'best_model.pt')
		else:
			patience_counter += 1
			if patience_counter >= patience:
				print(f"Early stopping after {epoch} epochs.")
				break

	# Cargar el mejor modelo
	model.load_state_dict(torch.load('best_model.pt'))
	return model

In [10]:
def evaluate(model, test_loader):
	model.eval()
	predictions = []
	targets = []
	with torch.no_grad():
		for batch in test_loader:
			sent1 = batch['sent1'].to(device)
			len1 = batch['len1'].to(device)
			sent2 = batch['sent2'].to(device)
			len2 = batch['len2'].to(device)
			labels = batch['label'].cpu().numpy()

			pred = model(sent1, len1, sent2, len2)
			predictions.extend(pred.cpu().numpy().flatten())
			targets.extend(labels.flatten())

	predictions = np.array(predictions)
	targets = np.array(targets)

	pearson, _ = pearsonr(predictions, targets)
	spearman, _ = spearmanr(predictions, targets)
	mse = np.mean((predictions - targets) ** 2)

	print("=== Test Set Evaluation ===")
	print(f"Pearson correlation: {pearson:.4f}")
	print(f"Spearman correlation: {spearman:.4f}")
	print(f"MSE: {mse:.6f}")